In [8]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpqk4d_f3w".


# Question 1
Create a simple hello world application in CUDA.
```
Hello from block: 0, thread: 0
Hello from block: 0, thread: 1
Hello from block: 1, thread: 0
Hello from block: 1, thread: 1
```

(the ordering of the above lines may vary; ordering differences do not indicate an incorrect result)


In [12]:
%%cuda
#include<stdio.h>
#include<cuda.h>

__global__ void mykernel()
{
    int a= blockIdx.x, b= threadIdx.x;
    printf("Hello from block: %d, thread: %d\n", a, b);
}

int main() {
    mykernel<<< 2,2 >>>();
    cudaDeviceSynchronize();
    return 0;
}

Hello from block: 0, thread: 0
Hello from block: 0, thread: 1
Hello from block: 1, thread: 0
Hello from block: 1, thread: 1



# Question 2

Write a complete vector add program from scratch.

Typical output when complete would look like this:
```
A[0] = 0.840188
B[0] = 0.394383
C[0] = 1.234571
```

In [24]:
%%cuda
#include<stdio.h>
#include<cuda.h>

#define N 2048
#define BLOCK_SIZE 1024

__global__ void mykernel( int *A, int *B, int *C)
{
    int id= threadIdx.x + blockIdx.x * blockDim.x;
    if ( id<N ) {
        C[id]= A[id] + B[id];
    }
}

int main() {
  // creating variables
  int *A, *B, *C, *dA, *dB, *dC;

  // allocating space for variables
  A= (int *)malloc( N*sizeof(int) );
  B= (int *)malloc( N*sizeof(int) );
  C= (int *)malloc( N*sizeof(int) );

  cudaMalloc ( (void**)&dA, N*sizeof(int) );
  cudaMalloc ( (void**)&dB, N*sizeof(int) );
  cudaMalloc ( (void**)&dC, N*sizeof(int) );

  // initializing and allocating space
  for ( int i=0;i<N;i++ ) {
      A[i]= 1; B[i]= 2;
  }

  // copying data from cpu to gpu
  cudaMemcpy( dA, A, N*sizeof(int), cudaMemcpyHostToDevice );
  cudaMemcpy( dB, B, N*sizeof(int), cudaMemcpyHostToDevice );

  int p= N/BLOCK_SIZE;
  mykernel<<< p+1 , BLOCK_SIZE >>>( dA, dB, dC);   cudaDeviceSynchronize();

  cudaMemcpy( C, dC, N*sizeof(int), cudaMemcpyDeviceToHost );

  for ( int i=0;i<N;i++ ) {
      printf("%d ", C[i]);
  }
  return 0;
}

3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 

## Question 3

Write a code to multiply 2 matrices given that you can multiply it naively using a loop inside the gpu kernel.

In [27]:
%%cuda
#include<stdio.h>
#include<cuda.h>

#define N 3
#define BLOCK_SIZE 1024

__global__ void mykernel( int *A, int *B, int *C)
{
    int id= blockIdx.x * BLOCK_SIZE + threadIdx.x;
    if ( id<N*N ) {
      int x= id/N, y= id%N;
      int sum=0;
      for ( int i=0;i<N;i++ ) {
        sum+= A[ x *N + i  ] * B[ i*N + y ];
      }
      C[id]= sum;
    }
}

int main() {
  // creating variables
  int *A, *B, *C, *dA, *dB, *dC;

  // allocating space for variables
  A= (int *)malloc( N*N*sizeof(int) );
  B= (int *)malloc( N*N*sizeof(int) );
  C= (int *)malloc( N*N*sizeof(int) );

  cudaMalloc ( (void**)&dA, N*N*sizeof(int) );
  cudaMalloc ( (void**)&dB, N*N*sizeof(int) );
  cudaMalloc ( (void**)&dC, N*N*sizeof(int) );

  // initializing and allocating space
  for ( int i=0;i<N*N;i++ ) {
      A[i]= 1; B[i]= 2;
  }

  // copying data from cpu to gpu
  cudaMemcpy( dA, A, N*N*sizeof(int), cudaMemcpyHostToDevice );
  cudaMemcpy( dB, B, N*N*sizeof(int), cudaMemcpyHostToDevice );

  int p= N/BLOCK_SIZE;
  mykernel<<< p+1 , BLOCK_SIZE >>>( dA, dB, dC);   cudaDeviceSynchronize();

  cudaMemcpy( C, dC, N*N*sizeof(int), cudaMemcpyDeviceToHost );

  for ( int i=0;i<N;i++ ) {
    for ( int j=0;j<N;j++ ) {
      printf("%d ", C[i*N + j]);
    }
    printf("\n");
  }
  return 0;
}

6 6 6 
6 6 6 
6 6 6 

